# 11 - ML Duration Forecast Scenarios

Notebook 10 built DCA technical modeled recoverable oil estimates.

This notebook adds a simple ML scenario forecast so we can compare DCA-style technical recovery against a machine-learning extension.

Important framing: long-horizon recursive ML forecasts are scenario outputs, not reserves estimates. They can drift because each predicted month becomes part of the feature history for later months.

This workflow remains oil-only. Gas forecasting remains excluded.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "martin_selected_30_monthly_production_normalized.csv"
DCA_OUTPUT_DIR = PROJECT_ROOT / "reports" / "dca_outputs"
ML_OUTPUT_DIR = PROJECT_ROOT / "reports" / "ml_outputs"

ML_OUTPUT_DIR.mkdir(exist_ok=True)

DATA_FILE.exists(), DCA_OUTPUT_DIR.exists(), ML_OUTPUT_DIR.exists()

## Scenario setup

We will use the same feature idea from notebook 07:

- target month on production
- last observed or predicted oil
- trailing 3-month average oil
- trailing 6-month average oil
- interval length proxy

For the long-horizon technical recovery scenario, we initialize each well with observed oil through month 33 and recursively forecast months 34-120.

In [ ]:
VALIDATION_END_MONTH = 33
ML_FORECAST_START_MONTH = 34
ML_FORECAST_END_MONTH = 120

TRAIN_TARGET_START_MONTH = 13
TRAIN_TARGET_END_MONTH = 33

In [ ]:
production = pd.read_csv(DATA_FILE)

production.shape

In [ ]:
oil_history = production[
    [
        "api8",
        "lease_name",
        "well_no",
        "field_name",
        "month_on_production",
        "oil_bbl",
        "interval_length_proxy_ft",
    ]
].copy()

oil_history.head()

## Build training rows

The model learns one-month-ahead oil from observed historical rows.

This is still a simple model. The purpose is not to claim ML is better for EUR; the purpose is to create a transparent ML scenario to compare against DCA.

In [ ]:
modeling_df = oil_history.sort_values(["api8", "month_on_production"]).copy()
well_groups = modeling_df.groupby("api8", group_keys=False)

modeling_df["target_month_on_production"] = well_groups["month_on_production"].shift(-1)
modeling_df["target_next_oil_bbl"] = well_groups["oil_bbl"].shift(-1)
modeling_df["last_observed_oil_bbl"] = modeling_df["oil_bbl"]
modeling_df["trailing_3mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)
modeling_df["trailing_6mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

modeling_df.head()

In [ ]:
feature_columns = [
    "target_month_on_production",
    "last_observed_oil_bbl",
    "trailing_3mo_avg_oil_bbl",
    "trailing_6mo_avg_oil_bbl",
    "interval_length_proxy_ft",
]

target_column = "target_next_oil_bbl"

In [ ]:
train_rows = modeling_df.dropna(subset=feature_columns + [target_column]).copy()
train_rows = train_rows[
    train_rows["target_month_on_production"].between(
        TRAIN_TARGET_START_MONTH,
        TRAIN_TARGET_END_MONTH,
    )
].copy()

train_rows.shape

## Fit a dependency-light ML model

To keep this notebook runnable in the basic project environment, we fit linear regression directly with NumPy.

This is the same family of model used in notebook 07, but without requiring scikit-learn.

In [ ]:
def fit_linear_regression_numpy(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    X_with_intercept = np.column_stack([np.ones(len(X)), X])
    coefficients, *_ = np.linalg.lstsq(X_with_intercept, y, rcond=None)
    return coefficients


def predict_linear_regression_numpy(X, coefficients):
    X = np.asarray(X, dtype=float)
    X_with_intercept = np.column_stack([np.ones(len(X)), X])
    return X_with_intercept @ coefficients

In [ ]:
linear_coefficients = fit_linear_regression_numpy(
    train_rows[feature_columns],
    train_rows[target_column],
)

linear_coefficients

## Recursive forecast function

For each well, the forecast starts from observed oil through month 33.

For each future month, we calculate features from the current history, predict that month's oil, floor negative predictions at zero, and then append the prediction to the history for the next month.

In [ ]:
def recursive_forecast_one_well(well_history, coefficients, start_month, end_month):
    well_history = well_history.sort_values("month_on_production").copy()
    oil_values = well_history["oil_bbl"].astype(float).tolist()
    rows = []

    interval_length = float(well_history["interval_length_proxy_ft"].iloc[0])

    for forecast_month in range(start_month, end_month + 1):
        last_observed = oil_values[-1]
        trailing_3mo = float(np.mean(oil_values[-3:]))
        trailing_6mo = float(np.mean(oil_values[-6:]))

        features = pd.DataFrame(
            [
                {
                    "target_month_on_production": forecast_month,
                    "last_observed_oil_bbl": last_observed,
                    "trailing_3mo_avg_oil_bbl": trailing_3mo,
                    "trailing_6mo_avg_oil_bbl": trailing_6mo,
                    "interval_length_proxy_ft": interval_length,
                }
            ]
        )

        ml_forecast = float(predict_linear_regression_numpy(features[feature_columns], coefficients)[0])
        ml_forecast = max(0.0, ml_forecast)

        naive_forecast = max(0.0, float(last_observed))
        trailing_3mo_forecast = max(0.0, trailing_3mo)
        trailing_6mo_forecast = max(0.0, trailing_6mo)

        rows.append(
            {
                "api8": well_history["api8"].iloc[0],
                "lease_name": well_history["lease_name"].iloc[0],
                "well_no": well_history["well_no"].iloc[0],
                "field_name": well_history["field_name"].iloc[0],
                "month_on_production": forecast_month,
                "linear_regression_ml_forecast_oil_bbl": ml_forecast,
                "naive_last_rate_forecast_oil_bbl": naive_forecast,
                "trailing_3mo_forecast_oil_bbl": trailing_3mo_forecast,
                "trailing_6mo_forecast_oil_bbl": trailing_6mo_forecast,
            }
        )

        oil_values.append(ml_forecast)

    return pd.DataFrame(rows)

In [ ]:
forecast_inputs = oil_history[
    oil_history["month_on_production"] <= VALIDATION_END_MONTH
].copy()

ml_forecast_monthly = pd.concat(
    [
        recursive_forecast_one_well(
            well_history=well_history,
            coefficients=linear_coefficients,
            start_month=ML_FORECAST_START_MONTH,
            end_month=ML_FORECAST_END_MONTH,
        )
        for _, well_history in forecast_inputs.groupby("api8")
    ],
    ignore_index=True,
)

ml_forecast_monthly.head()

In [ ]:
ml_forecast_monthly.shape

## ML technical recoverable oil table

Now we combine observed oil through month 33 with the ML forecast from months 34-120.

These are technical modeled recoverable oil scenario outputs. They are not SPE PRMS reserves estimates.

In [ ]:
historical_cumulative_oil = (
    forecast_inputs
    .groupby(["api8", "lease_name", "well_no", "field_name"], as_index=False)
    .agg(historical_cumulative_oil_bbl=("oil_bbl", "sum"))
)

historical_cumulative_oil.head()

In [ ]:
ml_forecast_cumulative = (
    ml_forecast_monthly
    .groupby(["api8", "lease_name", "well_no", "field_name"], as_index=False)
    .agg(
        linear_regression_ml_forecast_oil_bbl=("linear_regression_ml_forecast_oil_bbl", "sum"),
        naive_last_rate_forecast_oil_bbl=("naive_last_rate_forecast_oil_bbl", "sum"),
        trailing_3mo_forecast_oil_bbl=("trailing_3mo_forecast_oil_bbl", "sum"),
        trailing_6mo_forecast_oil_bbl=("trailing_6mo_forecast_oil_bbl", "sum"),
    )
)

ml_forecast_cumulative.head()

In [ ]:
ml_technical_recovery_table = historical_cumulative_oil.merge(
    ml_forecast_cumulative,
    on=["api8", "lease_name", "well_no", "field_name"],
    how="left",
)

ml_technical_recovery_table["well_name"] = (
    ml_technical_recovery_table["lease_name"]
    + " "
    + ml_technical_recovery_table["well_no"].astype(str)
)

for column in [
    "linear_regression_ml_forecast_oil_bbl",
    "naive_last_rate_forecast_oil_bbl",
    "trailing_3mo_forecast_oil_bbl",
    "trailing_6mo_forecast_oil_bbl",
]:
    total_column = column.replace("forecast_oil_bbl", "technical_recoverable_oil_bbl")
    ml_technical_recovery_table[total_column] = (
        ml_technical_recovery_table["historical_cumulative_oil_bbl"]
        + ml_technical_recovery_table[column]
    )

ml_technical_recovery_table.head()

In [ ]:
ml_technical_recovery_table = ml_technical_recovery_table[
    [
        "api8",
        "well_name",
        "lease_name",
        "well_no",
        "field_name",
        "historical_cumulative_oil_bbl",
        "linear_regression_ml_technical_recoverable_oil_bbl",
        "naive_last_rate_technical_recoverable_oil_bbl",
        "trailing_3mo_technical_recoverable_oil_bbl",
        "trailing_6mo_technical_recoverable_oil_bbl",
    ]
].sort_values("linear_regression_ml_technical_recoverable_oil_bbl", ascending=False)

ml_technical_recovery_table.round(0)

## Model-level ML scenario summary

This summary gives the cohort total for each ML or simple recursive baseline scenario.

In [ ]:
ml_model_level_summary = pd.DataFrame(
    [
        {
            "scenario": "linear_regression_ml",
            "technical_recoverable_oil_bbl": ml_technical_recovery_table[
                "linear_regression_ml_technical_recoverable_oil_bbl"
            ].sum(),
        },
        {
            "scenario": "naive_last_rate",
            "technical_recoverable_oil_bbl": ml_technical_recovery_table[
                "naive_last_rate_technical_recoverable_oil_bbl"
            ].sum(),
        },
        {
            "scenario": "trailing_3mo",
            "technical_recoverable_oil_bbl": ml_technical_recovery_table[
                "trailing_3mo_technical_recoverable_oil_bbl"
            ].sum(),
        },
        {
            "scenario": "trailing_6mo",
            "technical_recoverable_oil_bbl": ml_technical_recovery_table[
                "trailing_6mo_technical_recoverable_oil_bbl"
            ].sum(),
        },
    ]
)

ml_model_level_summary.round(0)

## Export notebook 11 outputs

In [ ]:
ML_DURATION_MONTHLY_FILE = ML_OUTPUT_DIR / "ml_duration_monthly_forecast_scenarios.csv"
ML_30_WELL_TABLE_FILE = ML_OUTPUT_DIR / "ml_30_well_technical_recoverable_oil_table.csv"
ML_MODEL_SUMMARY_FILE = ML_OUTPUT_DIR / "ml_duration_model_summary.csv"

ML_DURATION_MONTHLY_FILE, ML_30_WELL_TABLE_FILE, ML_MODEL_SUMMARY_FILE

In [ ]:
ml_forecast_monthly.to_csv(ML_DURATION_MONTHLY_FILE, index=False)
ml_technical_recovery_table.to_csv(ML_30_WELL_TABLE_FILE, index=False)
ml_model_level_summary.to_csv(ML_MODEL_SUMMARY_FILE, index=False)

In [ ]:
ML_DURATION_MONTHLY_FILE.exists(), ML_30_WELL_TABLE_FILE.exists(), ML_MODEL_SUMMARY_FILE.exists()

## Notebook 11 summary

This notebook created recursive ML technical recovery scenarios from months 34-120.

Key interpretation points:

- The ML scenario is initialized with observed oil through month 33.
- The forecast is recursive, so each predicted month becomes part of the feature history.
- Recursive ML forecasts can drift over long horizons.
- These outputs are technical modeled recoverable oil scenarios, not SPE PRMS reserves estimates.
- DCA remains the more natural framework for EUR/duration-style forecasting; ML is included here as an exploratory comparison.